# 中间件与浏览器跨域

学习目标：能观察中间件顺序，用请求标识关联日志，并正确区分 CORS 的响应读取限制、预检与服务端权限检查。

前置知识：HTTP 请求与响应、FastAPI 路由、日志、JavaScript 基础、浏览器开发者工具。

使用 Python 3.12、FastAPI 与支持 Fetch、AbortSignal.timeout 的现代浏览器，工作目录为 content/Web与应用开发/FastAPI。前面的代码直接在 Notebook 中运行，浏览器实验使用本地 API 8110 和页面服务 8111。

环境准备：[安装与运行说明](README.md)。

配套脚本：位于 scripts/11-middleware-and-cors，路径相对于本 Notebook。

1. [app.py](scripts/11-middleware-and-cors/app.py)：浏览器实验的 API、中间件、请求计数与临时 Cookie。
2. [index.html](scripts/11-middleware-and-cors/index.html)：发起跨域请求并显示响应与请求标识。

## 1 请求进入与响应返回的顺序

中间件（middleware）在路由处理前接收请求，并能修改路由生成的响应。await call_next(request) 把请求交给内层应用，返回后可处理响应头。

先定义中间件 A 和一条路由，用列表记录经过的位置。

In [1]:
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient

app = FastAPI()
order = []


@app.middleware("http")
async def middleware_a(request: Request, call_next):
    order.append("A 请求")
    response = await call_next(request)
    order.append("A 响应")
    return response


@app.get("/hello")
async def hello():
    order.append("路由")
    return {"message": "hello"}

C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


再添加中间件 B。后添加的中间件位于外层，所以它先处理请求、后处理响应。这两个中间件都在首次请求之前注册。

In [2]:
@app.middleware("http")
async def middleware_b(request: Request, call_next):
    order.append("B 请求")
    response = await call_next(request)
    order.append("B 响应")
    return response


with TestClient(app) as client:
    response = client.get("/hello")
print(response.json(), order)
assert order == ["B 请求", "A 请求", "路由", "A 响应", "B 响应"]

{'message': 'hello'} ['B 请求', 'A 请求', '路由', 'A 响应', 'B 响应']


## 2 给请求分配标识并记录日志

请求标识用于把一次请求的日志与响应对应起来。本例由服务器生成随机 UUID，并通过 X-Request-ID 响应头返回；它只用于追踪，不表示登录身份。

先设置一个本章专用的日志处理器（handler），把 INFO 消息显示到 Notebook。日志只写请求方法、路径、状态码和标识，不记录 Cookie、令牌或请求正文。

In [3]:
import logging
import sys
from uuid import uuid4

logger = logging.getLogger("fastapi.lesson11")
logger.setLevel(logging.INFO)
logger.propagate = False
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(message)s"))
logger.addHandler(handler)

在新的小应用中，用 request.state 保存本次请求的标识；路由可以读取它，中间件也把相同值写入响应头。这里的“响应”日志记录 call_next 返回的状态，不表示所有响应字节已发送完毕。

In [4]:
api = FastAPI()
hits = {"simple": 0, "json": 0}


@api.middleware("http")
async def identify_request(request: Request, call_next):
    request_id = uuid4().hex
    request.state.request_id = request_id
    logger.info("请求 %s %s id=%s", request.method, request.url.path, request_id)
    response = await call_next(request)
    response.headers["X-Request-ID"] = request_id
    logger.info("响应 status=%s id=%s", response.status_code, request_id)
    return response

增加 GET 和 JSON POST 两条路由。hits 只统计实际进入路由的次数，便于观察预检是否进入业务处理。

In [5]:
@api.get("/simple")
async def simple(request: Request):
    hits["simple"] += 1
    return {"request_id": request.state.request_id, "visits": hits["simple"]}


@api.post("/json")
async def json_echo(payload: dict[str, str]):
    hits["json"] += 1
    return payload

发出一次请求，把日志中的标识、响应头和路由返回的标识放在一起核对。UUID 每次运行会变化，不把某个具体值写成预期结果。

In [6]:
with TestClient(api) as client:
    response = client.get("/simple")
request_id = response.headers["X-Request-ID"]
print("响应标识：", request_id)
assert response.status_code == 200
assert request_id == response.json()["request_id"]

请求 GET /simple id=8f7ca3b52c7c498b9bac597ca8a07fb9


响应 status=200 id=8f7ca3b52c7c498b9bac597ca8a07fb9


响应标识： 8f7ca3b52c7c498b9bac597ca8a07fb9


## 3 源与 CORSMiddleware 配置

源（origin）由协议、主机和端口共同确定。http://127.0.0.1:8111 与 http://127.0.0.1:8110 端口不同；http://localhost:8111 与前者主机名不同，都是不同源。

CORS（跨源资源共享）通过响应头告诉浏览器哪些来源可以读取响应。下面允许 127.0.0.1:8111，保留 localhost:8111 作为对照；路径不属于源，不写进允许来源列表。

| 参数 | 中文名称／含义 |
| --- | --- |
| allow_origins | 允许的完整来源 |
| allow_methods | 允许跨域使用的 HTTP 方法 |
| allow_headers | 允许客户端请求携带的请求头 |
| allow_credentials | 是否允许跨域凭据请求 |
| expose_headers | 额外允许浏览器脚本读取的响应头 |
| max_age | 预检结果的缓存时间，单位为秒 |

使用凭据时显式列出来源、方法和请求头。把 CORSMiddleware 包在应用外层，也能为内层生成的错误响应应用 CORS 规则；预检会在此直接处理，不进入业务中间件与路由。

In [7]:
from fastapi.middleware.cors import CORSMiddleware

ALLOWED_ORIGIN = "http://127.0.0.1:8111"
DENIED_ORIGIN = "http://localhost:8111"
cors_app = CORSMiddleware(
    api,
    allow_origins=[ALLOWED_ORIGIN],
    allow_methods=["GET", "POST", "DELETE"],
    allow_headers=["Content-Type"],
    allow_credentials=True,
    expose_headers=["X-Request-ID"],
    max_age=0,  # 设为 0 秒，便于重复观察预检。
)

## 4 简单请求可能已被服务器处理

不携带额外请求头的 GET 通常属于不需要预检的“简单请求”。浏览器先发送请求，再检查响应是否允许当前页面读取；不允许来源并不意味着请求一定没有到达服务器。

TestClient 不执行浏览器的 CORS 限制。下面只检查服务器返回的头和实际路由计数：两种来源都得到服务端 200，但只有允许来源得到匹配的 Access-Control-Allow-Origin。

In [8]:
before = hits["simple"]
with TestClient(cors_app) as client:
    for origin in [ALLOWED_ORIGIN, DENIED_ORIGIN]:
        response = client.get("/simple", headers={"Origin": origin})
        allowed = response.headers.get("Access-Control-Allow-Origin")
        print(origin, "服务端状态：", response.status_code, "允许来源：", allowed)
        assert response.status_code == 200
        assert allowed == (ALLOWED_ORIGIN if origin == ALLOWED_ORIGIN else None)
print("实际进入 GET 路由次数：", hits["simple"] - before)
assert hits["simple"] - before == 2

请求 GET /simple id=615902f8de634e64a61a18b410dff2a6


响应 status=200 id=615902f8de634e64a61a18b410dff2a6


http://127.0.0.1:8111 服务端状态： 200 允许来源： http://127.0.0.1:8111
请求 GET /simple id=6f84d733b10d4cf9b2524ae7ffa1fae5


响应 status=200 id=6f84d733b10d4cf9b2524ae7ffa1fae5


http://localhost:8111 服务端状态： 200 允许来源： None
实际进入 GET 路由次数： 2


## 5 JSON 请求的预检

application/json 不属于简单请求允许的 Content-Type。浏览器会先发 OPTIONS 预检，带上 Origin、拟使用的方法 Access-Control-Request-Method，以及需要检查的请求头 Access-Control-Request-Headers。

CORSMiddleware 直接返回预检结果；允许时浏览器才发送实际 POST。下面手工检查两个来源的预检响应，确认 OPTIONS 没有增加 POST 路由计数。

In [9]:
before = hits["json"]
cases = [(ALLOWED_ORIGIN, 200), (DENIED_ORIGIN, 400)]
with TestClient(cors_app) as client:
    for origin, expected_status in cases:
        response = client.options("/json", headers={
            "Origin": origin,
            "Access-Control-Request-Method": "POST",
            "Access-Control-Request-Headers": "content-type",
        })
        print(origin, "预检状态：", response.status_code)
        assert response.status_code == expected_status
assert hits["json"] == before  # 预检由 CORS 层处理，不调用 POST 路由。

http://127.0.0.1:8111 预检状态： 200
http://localhost:8111 预检状态： 400


允许来源的实际请求仍需返回允许来源头，并暴露 X-Request-ID，页面才能通过 response.headers.get 读取这个自定义响应头。允许请求头与暴露响应头是两项不同配置。

In [10]:
with TestClient(cors_app) as client:
    response = client.post(
        "/json", headers={"Origin": ALLOWED_ORIGIN}, json={"message": "hello"},
    )
exposed = response.headers["Access-Control-Expose-Headers"]
print(response.json(), "可读响应头：", exposed)
assert response.status_code == 200
assert response.headers["Access-Control-Allow-Origin"] == ALLOWED_ORIGIN
assert "X-Request-ID" in exposed
assert hits["json"] == before + 1

请求 POST /json id=a953dd33c39b4b5eb3f0f9a787045df7


响应 status=200 id=a953dd33c39b4b5eb3f0f9a787045df7


{'message': 'hello'} 可读响应头： X-Request-ID


## 6 凭据传递不等于身份认证

Fetch 默认只在同源请求中携带 Cookie；跨源携带时使用 credentials: 'include'，并由服务端返回明确的允许来源和 Access-Control-Allow-Credentials: true。Cookie 的 SameSite 与浏览器策略仍然有效，include 不会覆盖这些限制；携带 Cookie 本身也不一定触发预检。

用一个固定、最长保留 300 秒的演示 Cookie 观察传递。它不是登录会话，接口只返回是否收到标记。真实身份和权限仍由服务器独立校验；CORS 不会阻止普通 HTTP 客户端直接调用公开接口，也不能替代 CSRF 防护。

先定义设置与读取标记的两个小接口。

In [11]:
from fastapi import Response


@api.post("/cookie")
async def set_demo_cookie(response: Response):
    response.set_cookie(
        "lesson11", "demo", max_age=300, httponly=True, samesite="lax",
    )
    return {"cookie_set": True}


@api.get("/credentials")
async def read_credentials(request: Request):
    return {"demo_cookie_received": request.cookies.get("lesson11") == "demo"}

删除接口清理同名 Cookie。先用 TestClient 检查 API 是否正确设置、接收和删除；浏览器的 omit/include 对照留给实际页面操作。

In [12]:
@api.delete("/cookie")
async def clear_demo_cookie(response: Response):
    response.delete_cookie("lesson11", httponly=True, samesite="lax")
    return {"cookie_cleared": True}


with TestClient(cors_app) as client:
    response = client.post("/cookie", headers={"Origin": ALLOWED_ORIGIN})
    assert response.headers["Access-Control-Allow-Credentials"] == "true"
    assert client.get("/credentials").json() == {"demo_cookie_received": True}
    client.delete("/cookie")
    assert client.get("/credentials").json() == {"demo_cookie_received": False}
print("演示 Cookie 设置、接收与删除完成")

请求 POST /cookie id=9974ea7be10e4873a96f50060095a993


响应 status=200 id=9974ea7be10e4873a96f50060095a993


请求 GET /credentials id=ace94f2beda046259c56d536d93486d7


响应 status=200 id=ace94f2beda046259c56d536d93486d7


请求 DELETE /cookie id=90df1fdd5be346e0aab2717e5d32c352


响应 status=200 id=90df1fdd5be346e0aab2717e5d32c352


请求 GET /credentials id=7f7c8b4791b3419188d823984e261b3c


响应 status=200 id=7f7c8b4791b3419188d823984e261b3c


演示 Cookie 设置、接收与删除完成


Notebook 中的日志演示结束后，移除本章添加的处理器。配套服务使用 Uvicorn 的日志输出，不依赖当前内核的日志配置。

In [13]:
logger.removeHandler(handler)
handler.close()
print("已移除本章日志处理器")

已移除本章日志处理器


## 7 在浏览器中对照允许与拒绝来源

配套 app.py 整理了上述请求标识、CORS 与 Cookie 示例，额外提供 /stats 返回实际 GET、POST 路由计数。index.html 的按钮通过 Fetch 请求 API，使用 textContent 显示响应，等待上限为 5000 毫秒。

两个页面地址由同一个静态服务提供。凭据实验在允许页面完成：页面与 API 都使用 http://127.0.0.1，端口不同使它们跨源，但仍是同站请求，便于观察 SameSite=Lax 的 Cookie。

Step 1：在课程目录的终端启动 API。

```bash
python -m uvicorn app:app --app-dir scripts/11-middleware-and-cors --host 127.0.0.1 --port 8110
```

Step 2：在课程目录的另一个终端启动静态页面服务。

```bash
python -m http.server 8111 --bind 127.0.0.1 --directory scripts/11-middleware-and-cors
```

Step 3：打开 http://127.0.0.1:8111/index.html，并在开发者工具的 Network 中观察请求。

点击“简单 GET”，检查页面能显示响应和请求标识；点击“JSON POST”，检查 OPTIONS 成功后出现 POST。响应头中的 X-Request-ID 可与 API 终端中同一请求的日志对应。

Step 4：在新标签页打开 http://localhost:8111/index.html，保留允许页面用于读取计数。

先在允许页面点击“读取服务器计数”，再在拒绝页面点击“简单 GET”，最后回到允许页面再次读取计数。检查 GET 计数增加，拒绝页面却无法读取响应。再对照“JSON POST”：拒绝页面的 OPTIONS 返回 400，实际 POST 没有发送，POST 计数不增加。

浏览器脚本得到的错误可能同时涵盖网络与 CORS 失败，具体原因查看 Console 和 Network。把 Fetch 改为 no-cors 会得到不可读取正文和响应头的 opaque 响应，不能用它修复读取问题。

![拒绝来源页面的跨域读取失败提示](image/11-cors-denied.png)

Step 5：在允许页面点击“演示凭据”。

页面设置短期测试 Cookie，分别用 omit 与 include 请求 /credentials；检查前者为 false、后者为 true。按钮在 finally 中删除测试 Cookie，随后检查 Application 的 Cookies 中已无 lesson11。页面只显示是否收到标记，不把它称为认证结果。

![允许来源页面的 Cookie 传递与删除结果](image/11-cors-allowed.png)

Step 6：完成后分别在 API 与静态服务终端按 Ctrl+C，关闭两个服务。

## 本章小结

（1）后添加的中间件先处理请求、后处理响应。请求标识把应用日志与响应联系起来，不承担认证作用。

（2）CORS 决定浏览器能否跨源读取响应。简单请求可能已经被执行；需要预检的请求在预检失败后不会继续发送实际请求。

（3）凭据、来源与可读响应头分别配置。浏览器 Cookie 策略、服务器身份权限检查和 CSRF 防护仍各自适用。

## 练习

1. 在第 1 节首次请求之前再添加中间件 C，记录同样的请求与响应标记。重启内核并按顺序运行，检查 C 位于请求序列最前、响应序列最后。

2. 在配套服务中暂时删除 expose_headers 中的 X-Request-ID 并重启 API。从允许页面发出简单 GET，检查正文可读、Network 中仍有该响应头，而 JavaScript 读取该头得到 null。完成后恢复配置并重启。

3. 在允许页面的 JSON 请求中增加 X-Lesson 请求头，先保持服务端 allow_headers 不变，检查预检被拒绝且 POST 计数不增。再显式允许 X-Lesson 并重启 API，检查预检和实际 POST 都成功。

提示：

（1）中间件配置应在首次处理请求前完成。

（2）第 2 题区分 Network 能看到的头与页面脚本可读的头。

（3）第 3 题观察预检中的 Access-Control-Request-Headers；改动后恢复示例文件。

## 参考与引用来源

1. **FastAPI 官方文档**：[Middleware](https://fastapi.tiangolo.com/tutorial/middleware/#multiple-middleware-execution-order)，请求与响应流程、叠加顺序、自定义响应头；[CORS](https://fastapi.tiangolo.com/tutorial/cors/)，源、CORSMiddleware 参数和预检；[Response Cookies](https://fastapi.tiangolo.com/advanced/response-cookies/)，通过响应对象设置 Cookie。

2. **Starlette 官方文档**：[CORSMiddleware 与 Global Enforcement](https://starlette.dev/middleware/#corsmiddleware)，预检拦截、简单请求处理及外层包装；[Requests](https://starlette.dev/requests/#other-state)，request.state；[Response Cookie 接口](https://starlette.dev/responses/#set-cookie)，set_cookie 与 delete_cookie。

3. **MDN Web Docs**：[CORS](https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/CORS) 的 Simple requests、Preflighted requests、Requests with credentials；[Using Fetch](https://developer.mozilla.org/en-US/docs/Web/API/Fetch_API/Using_Fetch#including_credentials)，Cookie、凭据模式、no-cors 和错误处理；[Set-Cookie](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Headers/Set-Cookie#samesitesamesite-value) 与 [Site](https://developer.mozilla.org/en-US/docs/Glossary/Site)，SameSite、同站与跨源的区别；[AbortSignal.timeout](https://developer.mozilla.org/en-US/docs/Web/API/AbortSignal/timeout_static)，等待取消与毫秒参数；[textContent](https://developer.mozilla.org/en-US/docs/Web/API/Node/textContent)，以文本显示结果。

4. **WHATWG Fetch 标准**：[HTTP CORS protocol](https://fetch.spec.whatwg.org/#http-cors-protocol)，跨源响应共享、凭据条件与 CORS 检查。

5. **Python 3.12 官方文档**：[uuid.uuid4](https://docs.python.org/3.12/library/uuid.html#uuid.uuid4)，随机 UUID 与 hex；[Logging HOWTO](https://docs.python.org/3.12/howto/logging.html#configuring-logging) 与 [Logging API](https://docs.python.org/3.12/library/logging.html#logging.Logger.removeHandler)，记录器、处理器、消息格式与清理；[http.server](https://docs.python.org/3.12/library/http.server.html#command-line-interface)，本地服务的端口、绑定地址与目录参数。

6. **Uvicorn 官方文档**：[Settings](https://uvicorn.dev/settings/#application)，应用入口、app-dir、host 与 port；[Logging](https://uvicorn.dev/concepts/logging/)，uvicorn.error 与 uvicorn.access 记录器及默认输出配置。